In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

import os
import sys

project_path = os.path.join(os.getcwd(),'..','..')
sys.path.append(project_path)

from utils.transformations import reusable

## DimUser

#### AUTOLOADER

# STEP 2: Delete the corrupted checkpoint
dbutils.fs.rm("abfss://silver@atanustoragespotify.dfs.core.windows.net/DimUser/autoloader/checkpoint", recurse=True)
print("Checkpoint deleted - Starting fresh!")



In [0]:
schema_loc = "abfss://silver@atanustoragespotify.dfs.core.windows.net/DimUser/autoloader/schemaLocation"
checkpoint_loc = "abfss://silver@atanustoragespotify.dfs.core.windows.net/DimUser/autoloader/checkpoint"

df_user = spark.readStream.format("cloudFiles")\
                          .option("cloudFiles.format", "parquet")\
                          .option("cloudFiles.schemaLocation", schema_loc)\
                          .option("schemaEvolutionMode", "addNewColumns")\
                          .load("abfss://bronze@atanustoragespotify.dfs.core.windows.net/DimUser")\
                          .withColumn("user_name", upper(col("user_name")))


In [0]:
df_user.writeStream \
  .format("delta") \
  .outputMode("append") \
  .option("path", "abfss://silver@atanustoragespotify.dfs.core.windows.net/DimUser/raw_data")\
  .option("checkpointLocation", checkpoint_loc) \
  .trigger(availableNow=True)\
  .start()

In [0]:
df_user_trans = spark.read.format("delta") \
     .load("abfss://silver@atanustoragespotify.dfs.core.windows.net/DimUser/raw_data") \

df_user_trans.display()


In [0]:
df_user_obj = reusable()

df_user_trans = df_user_obj.drop_columns(df_user_trans,['_rescued_data'])
df_user_trans = df_user_trans.dropDuplicates(['user_id'])
display(df_user_trans)

In [0]:
df_user_trans.write.format("delta") \
    .mode("append") \
    .option("path","abfss://silver@atanustoragespotify.dfs.core.windows.net/DimUser/clean_data")\
    .saveAsTable("spotify_cata.silver.DimUser")



## DimArtist

In [0]:
schema_loc = "abfss://silver@atanustoragespotify.dfs.core.windows.net/DimArtist/autoloader/schemaLocation"
checkpoint_loc = "abfss://silver@atanustoragespotify.dfs.core.windows.net/DimArtist/autoloader/checkpoint"

df_artist = spark.readStream.format("cloudFiles")\
                          .option("cloudFiles.format", "parquet")\
                          .option("cloudFiles.schemaLocation", schema_loc)\
                          .option("schemaEvolutionMode", "addNewColumns")\
                          .load("abfss://bronze@atanustoragespotify.dfs.core.windows.net/DimArtist")\
                          


In [0]:
df_artist.writeStream \
  .format("delta") \
  .outputMode("append") \
  .option("path", "abfss://silver@atanustoragespotify.dfs.core.windows.net/DimArtist/raw_data")\
  .option("checkpointLocation", checkpoint_loc) \
  .trigger(availableNow=True)\
  .start()

In [0]:
df_artist_trans = spark.read.format("delta") \
     .load("abfss://silver@atanustoragespotify.dfs.core.windows.net/DimArtist/raw_data") \

df_artist_trans.display()


In [0]:
df_artist_obj = reusable()

df_artist_trans = df_artist_obj.drop_columns(df_artist_trans,['_rescued_data'])
df_artist_trans = df_artist_trans.dropDuplicates(['artist_id'])
display(df_artist_trans)

In [0]:
df_artist_trans.write.format("delta") \
    .mode("append") \
    .option("path","abfss://silver@atanustoragespotify.dfs.core.windows.net/DimArtist/clean_data")\
    .saveAsTable("spotify_cata.silver.DimArtist")



## DimTrack

In [0]:
schema_loc = "abfss://silver@atanustoragespotify.dfs.core.windows.net/DimTrack/autoloader/schemaLocation"
checkpoint_loc = "abfss://silver@atanustoragespotify.dfs.core.windows.net/DimTrack/autoloader/checkpoint"

df_track = spark.readStream.format("cloudFiles")\
                          .option("cloudFiles.format", "parquet")\
                          .option("cloudFiles.schemaLocation", schema_loc)\
                          .option("schemaEvolutionMode", "addNewColumns")\
                          .load("abfss://bronze@atanustoragespotify.dfs.core.windows.net/DimTrack")\
                          


In [0]:
df_track.writeStream \
  .format("delta") \
  .outputMode("append") \
  .option("path", "abfss://silver@atanustoragespotify.dfs.core.windows.net/DimTrack/raw_data")\
  .option("checkpointLocation", checkpoint_loc) \
  .trigger(availableNow=True)\
  .start()

In [0]:
df_track_trans = spark.read.format("delta") \
     .load("abfss://silver@atanustoragespotify.dfs.core.windows.net/DimTrack/raw_data") \

df_track_trans.display()

In [0]:
df_track_trans = df_track_trans.withColumn("durationFlag",when(col('duration_sec') < 150, 'low')\
                                                         .when((col('duration_sec') >= 150) & (col('duration_sec') < 300), 'medium')\
                                                         .when(col('duration_sec') >= 300, 'long')\
                                                         .otherwise('short'))
                                                         

df_track_trans = df_track_trans.withColumn("track_name",regexp_replace(col("track_name"), "-", " "))

df_track_obj = reusable()
df_track_trans = df_track_obj.drop_columns(df_track_trans,['_rescued_data'])

df_track_trans.display()

In [0]:
df_track_trans.write.format("delta") \
    .mode("append") \
    .option("path","abfss://silver@atanustoragespotify.dfs.core.windows.net/DimTrack/clean_data")\
    .saveAsTable("spotify_cata.silver.DimTrack")


## DimDate

In [0]:
schema_loc = "abfss://silver@atanustoragespotify.dfs.core.windows.net/DimDate/autoloader/schemaLocation"
checkpoint_loc = "abfss://silver@atanustoragespotify.dfs.core.windows.net/DimDate/autoloader/checkpoint"

df_date = spark.readStream.format("cloudFiles")\
                          .option("cloudFiles.format", "parquet")\
                          .option("cloudFiles.schemaLocation", schema_loc)\
                          .option("schemaEvolutionMode", "addNewColumns")\
                          .load("abfss://bronze@atanustoragespotify.dfs.core.windows.net/DimDate")

In [0]:
df_date.writeStream \
  .format("delta") \
  .outputMode("append") \
  .option("path", "abfss://silver@atanustoragespotify.dfs.core.windows.net/DimDate/raw_data")\
  .option("checkpointLocation", checkpoint_loc) \
  .trigger(availableNow=True)\
  .start()

In [0]:
df_date_trans = spark.read.format("delta") \
     .load("abfss://silver@atanustoragespotify.dfs.core.windows.net/DimDate/raw_data") \

df_date_trans.display()

In [0]:
df_date_obj = reusable()

df_date_trans = df_date_obj.drop_columns(df_date_trans,['_rescued_data'])

display(df_date_trans)

In [0]:
df_date_trans.write.format("delta") \
    .mode("append") \
    .option("path","abfss://silver@atanustoragespotify.dfs.core.windows.net/DimDate/clean_data")\
    .saveAsTable("spotify_cata.silver.DimDate")


## FactStream

In [0]:
schema_loc = "abfss://silver@atanustoragespotify.dfs.core.windows.net/FactStream/autoloader/schemaLocation"
checkpoint_loc = "abfss://silver@atanustoragespotify.dfs.core.windows.net/FactStream/autoloader/checkpoint"

df_fact = spark.readStream.format("cloudFiles")\
                          .option("cloudFiles.format", "parquet")\
                          .option("cloudFiles.schemaLocation", schema_loc)\
                          .option("schemaEvolutionMode", "addNewColumns")\
                          .load("abfss://bronze@atanustoragespotify.dfs.core.windows.net/FactStream")\
                          


In [0]:
df_fact.writeStream \
  .format("delta") \
  .outputMode("append") \
  .option("path", "abfss://silver@atanustoragespotify.dfs.core.windows.net/FactStream/raw_data")\
  .option("checkpointLocation", checkpoint_loc) \
  .trigger(availableNow=True)\
  .start()

In [0]:
df_fact_trans = spark.read.format("delta") \
     .load("abfss://silver@atanustoragespotify.dfs.core.windows.net/FactStream/raw_data") \

df_fact_trans.display()

In [0]:
df_fact_obj = reusable()

df_fact_trans = df_fact_obj.drop_columns(df_fact_trans,['_rescued_data'])

display(df_fact_trans)

In [0]:
df_fact_trans.write.format("delta") \
    .mode("append") \
    .option("path","abfss://silver@atanustoragespotify.dfs.core.windows.net/FactStream/clean_data")\
    .saveAsTable("spotify_cata.silver.FactStream")